In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1tLqojRVmjzY0R1mRWrLoXO3EnV74pzDn", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))


# The LayerNorm Problem: Measuring Activation Memory Waste in Tensor Parallelism

*Notebook 1 of the Sequence Parallelism series — Vizuara GPU Programming Course*

*Estimated time: 40-50 minutes*

---

In this notebook, we will discover a surprising memory inefficiency that arises when you use tensor parallelism (TP) to distribute a transformer model across GPUs. While TP successfully splits the heavy MLP and attention weight matrices, the **LayerNorm** and **dropout** operations still force every GPU to hold **full-size** activation tensors. We will measure this waste precisely, understand why it happens, and quantify how much memory is being thrown away.

In [ ]:
#@title 🎧 Listen: Ai Assistant
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_ai_assistant.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/gpu-programming/practice/7/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

In [ ]:
#@title 🎧 Code Walkthrough: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU detected. This notebook runs on CPU too (simulated parallelism).")

print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

torch.manual_seed(42)
np.random.seed(42)

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Recap Tp Splits
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_recap_tp_splits.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. Recap: What Tensor Parallelism Actually Splits

Tensor parallelism distributes a single transformer layer across $N$ GPUs by splitting the **weight matrices** of the MLP and attention layers:

- **Column-parallel**: The first MLP linear layer (and Q/K/V projections) is split along the output dimension. Each GPU stores and computes $h/N$ output features.
- **Row-parallel**: The second MLP linear layer (and output projection) is split along the input dimension. Each GPU holds $h/N$ input features and produces partial sums that are AllReduced.

For the MLP and attention internals, each GPU stores activations of shape $(b, s, h/N)$ — properly split. But what about the operations **between** the tensor-parallel regions?

Let us build a simulated transformer layer and measure the memory for every component.

In [ ]:
#@title 🎧 Transition: Building Layer
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_building_layer.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building a Transformer Layer with Tensor Parallelism

We will simulate $N = 4$ GPUs on a single device. For the tensor-parallel regions, each "GPU" stores $h/N$ hidden dimensions. For the non-tensor-parallel regions (LayerNorm, dropout, residual), we will see what happens.

In [ ]:
#@title 🎧 Code Walkthrough: Model Config
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_model_config.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Model configuration (roughly GPT-3 13B scale)
HIDDEN_DIM = 5120       # h
SEQ_LEN = 2048          # s
BATCH_SIZE = 1          # b (micro-batch)
NUM_HEADS = 40
TP_DEGREE = 4           # N = 4 simulated GPUs
MLP_RATIO = 4           # MLP intermediate size = 4h

print(f"Configuration:")
print(f"  Hidden dimension (h): {HIDDEN_DIM}")
print(f"  Sequence length (s):  {SEQ_LEN}")
print(f"  Batch size (b):       {BATCH_SIZE}")
print(f"  TP degree (N):        {TP_DEGREE}")
print(f"  MLP intermediate:     {HIDDEN_DIM * MLP_RATIO}")
print(f"\nFull activation shape:   ({BATCH_SIZE}, {SEQ_LEN}, {HIDDEN_DIM})")
print(f"TP-split activation:     ({BATCH_SIZE}, {SEQ_LEN}, {HIDDEN_DIM // TP_DEGREE})")

In [ ]:
#@title 🎧 Code Walkthrough: Memory Functions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_memory_functions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def bytes_to_mb(num_bytes):
    """Convert bytes to megabytes."""
    return num_bytes / (1024 ** 2)

def activation_memory(shape, dtype_bytes=2):
    """Compute activation memory in bytes for a given shape (BF16 = 2 bytes)."""
    elements = 1
    for dim in shape:
        elements *= dim
    return elements * dtype_bytes

# Full activation: (b, s, h)
full_act_bytes = activation_memory((BATCH_SIZE, SEQ_LEN, HIDDEN_DIM))

# TP-split activation: (b, s, h/N)
tp_act_bytes = activation_memory((BATCH_SIZE, SEQ_LEN, HIDDEN_DIM // TP_DEGREE))

print(f"Full activation (b, s, h):     {bytes_to_mb(full_act_bytes):.2f} MB")
print(f"TP-split activation (b, s, h/N): {bytes_to_mb(tp_act_bytes):.2f} MB")
print(f"\nRatio: full / split = {full_act_bytes / tp_act_bytes:.1f}x")

In [ ]:
#@title 🎧 Listen: Why Layernorm Needs Full H
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_why_layernorm_needs_full_h.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Why LayerNorm Needs the Full Hidden Dimension

LayerNorm computes the mean and variance **across the hidden dimension** for each token independently:

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

where $\mu = \frac{1}{h} \sum_{i=1}^{h} x_i$ and $\sigma^2 = \frac{1}{h} \sum_{i=1}^{h} (x_i - \mu)^2$.

If the hidden dimension were split across GPUs, each GPU would only see $h/N$ features and would compute the **wrong** mean and variance. Let us verify this numerically.

In [ ]:
#@title 🎧 Code Walkthrough: Layernorm Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_layernorm_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstrate: LayerNorm CANNOT be computed on partial hidden dimensions

h = 16  # small hidden dim for demonstration
x = torch.randn(1, 4, h)  # (batch=1, seq=4, hidden=16)

# Correct LayerNorm on full hidden dim
ln = nn.LayerNorm(h)
with torch.no_grad():
    ln.weight.fill_(1.0)
    ln.bias.fill_(0.0)

correct_output = ln(x)

# Wrong: apply LayerNorm to each half separately
x_half1 = x[:, :, :h//2]   # first h/2 features
x_half2 = x[:, :, h//2:]   # second h/2 features

ln_partial = nn.LayerNorm(h // 2)
with torch.no_grad():
    ln_partial.weight.fill_(1.0)
    ln_partial.bias.fill_(0.0)

wrong_half1 = ln_partial(x_half1)
wrong_half2 = ln_partial(x_half2)
wrong_output = torch.cat([wrong_half1, wrong_half2], dim=-1)

# Compare
max_error = (correct_output - wrong_output).abs().max().item()
print(f"Max difference between correct and split LayerNorm: {max_error:.6f}")
print(f"\nCorrect output (token 0): {correct_output[0, 0, :8].tolist()}")
print(f"Wrong output   (token 0): {wrong_output[0, 0, :8].tolist()}")
print(f"\nConclusion: LayerNorm on split hidden dims gives WRONG results!")
print(f"Each GPU must have the FULL hidden dimension for LayerNorm.")

In [ ]:
#@title 🎧 Transition: Simulating Tp Layer
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_simulating_tp_layer.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Simulating a TP Transformer Layer and Measuring Memory

Let us build a simulated TP transformer layer where we track exactly which activations are stored on each "GPU." We will separate the TP regions (split) from the non-TP regions (replicated).

In [ ]:
#@title 🎧 Code Walkthrough: Simulated Tp Layer Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_simulated_tp_layer_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SimulatedTPTransformerLayer:
    """Simulates one transformer layer with tensor parallelism.
    
    Tracks which activations are stored and their shapes to compute
    exact memory usage per GPU.
    """
    
    def __init__(self, hidden_dim, num_heads, tp_degree, mlp_ratio=4):
        self.h = hidden_dim
        self.N = tp_degree
        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio
        self.head_dim = hidden_dim // num_heads
        self.heads_per_gpu = num_heads // tp_degree
        
    def trace_activations(self, batch_size, seq_len):
        """Trace all stored activations through the forward pass.
        
        Returns a dict mapping activation name -> (shape, category)
        where category is 'tp' (split) or 'non-tp' (replicated).
        """
        b, s, h, N = batch_size, seq_len, self.h, self.N
        
        activations = {}
        
        # === Attention block ===
        # LayerNorm 1 input: must be full (b, s, h) — needs full hidden dim
        activations['ln1_input'] = ((b, s, h), 'non-tp')
        
        # After AllReduce, before LayerNorm: full tensor
        # LayerNorm output feeds into TP attention
        
        # Q, K, V projections (column-parallel): each GPU has heads_per_gpu heads
        activations['q_proj'] = ((b, s, self.heads_per_gpu, self.head_dim), 'tp')
        activations['k_proj'] = ((b, s, self.heads_per_gpu, self.head_dim), 'tp')
        activations['v_proj'] = ((b, s, self.heads_per_gpu, self.head_dim), 'tp')
        
        # Attention scores: (b, heads_per_gpu, s, s)
        activations['attn_scores'] = ((b, self.heads_per_gpu, s, s), 'tp')
        
        # Attention output before output projection: (b, s, heads_per_gpu * head_dim)
        activations['attn_out'] = ((b, s, self.heads_per_gpu * self.head_dim), 'tp')
        
        # After row-parallel output proj + AllReduce: dropout receives full tensor
        activations['dropout1_input'] = ((b, s, h), 'non-tp')
        
        # Dropout mask: same shape as input
        activations['dropout1_mask'] = ((b, s, h), 'non-tp')  # 1 byte per element
        
        # Residual connection stored for backward
        activations['residual1'] = ((b, s, h), 'non-tp')
        
        # === MLP block ===
        # LayerNorm 2 input: full tensor
        activations['ln2_input'] = ((b, s, h), 'non-tp')
        
        # Column-parallel first linear: (b, s, 4h/N)
        activations['mlp_up'] = ((b, s, self.mlp_ratio * h // N), 'tp')
        
        # GeLU activation stored for backward: same shape
        activations['mlp_gelu'] = ((b, s, self.mlp_ratio * h // N), 'tp')
        
        # After row-parallel second linear + AllReduce: full tensor
        activations['dropout2_input'] = ((b, s, h), 'non-tp')
        activations['dropout2_mask'] = ((b, s, h), 'non-tp')
        
        return activations


layer = SimulatedTPTransformerLayer(HIDDEN_DIM, NUM_HEADS, TP_DEGREE)
activations = layer.trace_activations(BATCH_SIZE, SEQ_LEN)

print(f"{'Activation':<25} {'Shape':<35} {'Category':<10} {'MB (BF16)':<10}")
print("-" * 85)

total_tp = 0
total_non_tp = 0

for name, (shape, category) in activations.items():
    # Dropout masks are 1 byte, everything else is 2 bytes (BF16)
    dtype_bytes = 1 if 'mask' in name else 2
    mem = activation_memory(shape, dtype_bytes)
    mb = bytes_to_mb(mem)
    
    if category == 'tp':
        total_tp += mem
    else:
        total_non_tp += mem
    
    shape_str = str(shape)
    print(f"{name:<25} {shape_str:<35} {category:<10} {mb:<10.2f}")

print(f"\n{'='*85}")
print(f"Total TP (properly split) activations:     {bytes_to_mb(total_tp):.2f} MB per GPU")
print(f"Total non-TP (REPLICATED) activations:     {bytes_to_mb(total_non_tp):.2f} MB per GPU")
print(f"Total activation memory per GPU:           {bytes_to_mb(total_tp + total_non_tp):.2f} MB")
print(f"\nNon-TP fraction: {total_non_tp / (total_tp + total_non_tp) * 100:.1f}% of total!")

In [ ]:
#@title 🎧 Transition: Visualizing Waste Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_visualizing_waste_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Visualizing the Memory Waste

Let us create a clear visualization showing the memory breakdown for each simulated GPU.

In [ ]:
#@title 🎧 What to Look For: Visualization Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_visualization_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize memory breakdown per GPU

tp_mb = bytes_to_mb(total_tp)
non_tp_mb = bytes_to_mb(total_non_tp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked bar for each GPU
ax = axes[0]
gpus = [f'GPU {i}' for i in range(TP_DEGREE)]
x = np.arange(TP_DEGREE)

bars1 = ax.bar(x, [tp_mb] * TP_DEGREE, label='TP activations (split)',
               color='#2196F3', alpha=0.8, width=0.6)
bars2 = ax.bar(x, [non_tp_mb] * TP_DEGREE, bottom=[tp_mb] * TP_DEGREE,
               label='Non-TP activations (REPLICATED)', color='#F44336', alpha=0.8, width=0.6)

ax.set_ylabel('Activation Memory (MB)', fontsize=12)
ax.set_title('Per-GPU Activation Memory\n(Standard Tensor Parallelism)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(gpus)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i in range(TP_DEGREE):
    ax.text(i, tp_mb / 2, f'{tp_mb:.0f} MB', ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')
    ax.text(i, tp_mb + non_tp_mb / 2, f'{non_tp_mb:.0f} MB', ha='center',
            va='center', fontsize=9, fontweight='bold', color='white')

# Right: pie chart showing waste
ax = axes[1]
total_system_non_tp = non_tp_mb * TP_DEGREE
ideal_non_tp = non_tp_mb  # if distributed, only 1x needed total
wasted = total_system_non_tp - ideal_non_tp

sizes = [ideal_non_tp, wasted, tp_mb * TP_DEGREE]
labels = [f'Needed non-TP\n({ideal_non_tp:.0f} MB)',
          f'WASTED non-TP\n({wasted:.0f} MB)',
          f'TP activations\n({tp_mb * TP_DEGREE:.0f} MB)']
colors = ['#4CAF50', '#F44336', '#2196F3']
explode = (0, 0.1, 0)

ax.pie(sizes, labels=labels, colors=colors, explode=explode,
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
ax.set_title(f'System-Wide Memory Allocation\n(Total across {TP_DEGREE} GPUs)', fontsize=13)

plt.tight_layout()
plt.show()

print(f"\nSystem-wide waste: {wasted:.0f} MB of non-TP activations are redundant copies!")
print(f"Each GPU holds {non_tp_mb:.0f} MB of non-TP activations, but only {non_tp_mb:.0f} MB total is needed.")
print(f"Waste factor: {TP_DEGREE}x (N = {TP_DEGREE})")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_todo_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Your Turn: Compute Memory Waste at Different Scales

### TODO: Complete the function to compute memory waste for different model configurations

Calculate the non-TP activation memory waste for various model sizes and TP degrees.

In [ ]:
def compute_memory_waste(hidden_dim, seq_len, batch_size, num_layers, tp_degree, dtype_bytes=2):
    """Compute the activation memory waste from replicated non-TP activations.
    
    In a standard transformer layer with TP, the non-TP activations include:
    - 2 LayerNorm inputs: each (b, s, h)
    - 2 dropout inputs: each (b, s, h)  
    - 2 dropout masks: each (b, s, h) at 1 byte
    - 1 residual: (b, s, h)
    
    Args:
        hidden_dim: h
        seq_len: s
        batch_size: b
        num_layers: L (number of transformer layers)
        tp_degree: N (number of GPUs)
        dtype_bytes: bytes per element (2 for BF16)
    
    Returns:
        dict with:
          - 'non_tp_per_gpu_per_layer_mb': MB of non-TP activations per GPU per layer
          - 'non_tp_per_gpu_total_mb': total across all layers
          - 'ideal_per_gpu_total_mb': what it would be if distributed (divided by N)
          - 'wasted_per_gpu_mb': excess memory per GPU
    """
    b, s, h, N, L = batch_size, seq_len, hidden_dim, tp_degree, num_layers
    
    # ============ TODO ============
    # Step 1: Count the number of full (b, s, h) activations stored per layer
    #         Hint: 2 LN inputs + 2 dropout inputs + 1 residual = 5 tensors at dtype_bytes
    #               2 dropout masks at 1 byte each
    num_full_tensors = ???  # YOUR CODE: how many (b, s, h) tensors at dtype_bytes?
    num_mask_tensors = ???  # YOUR CODE: how many (b, s, h) masks at 1 byte?
    
    # Step 2: Compute bytes per layer per GPU for non-TP activations
    elements_per_tensor = ???  # YOUR CODE: b * s * h
    non_tp_bytes_per_layer = ???  # YOUR CODE: full tensors + mask tensors
    
    # Step 3: Compute total across all layers
    non_tp_bytes_total = ???  # YOUR CODE
    
    # Step 4: Compute ideal (if we could split by N)
    ideal_bytes_total = ???  # YOUR CODE
    
    # Step 5: Waste = actual - ideal
    wasted_bytes = ???  # YOUR CODE
    # ==============================
    
    return {
        'non_tp_per_gpu_per_layer_mb': bytes_to_mb(non_tp_bytes_per_layer),
        'non_tp_per_gpu_total_mb': bytes_to_mb(non_tp_bytes_total),
        'ideal_per_gpu_total_mb': bytes_to_mb(ideal_bytes_total),
        'wasted_per_gpu_mb': bytes_to_mb(wasted_bytes),
    }


# Test with our current config
result = compute_memory_waste(HIDDEN_DIM, SEQ_LEN, BATCH_SIZE, 40, TP_DEGREE)
print(f"Non-TP memory per GPU per layer: {result['non_tp_per_gpu_per_layer_mb']:.2f} MB")
print(f"Non-TP memory per GPU total ({40} layers): {result['non_tp_per_gpu_total_mb']:.2f} MB")
print(f"Ideal (distributed):  {result['ideal_per_gpu_total_mb']:.2f} MB")
print(f"Wasted per GPU:       {result['wasted_per_gpu_mb']:.2f} MB")

---


In [ ]:
#@title 🎧 Listen: Stop And Think
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_stop_and_think.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Stop and Think

Before looking at the solution, consider:
1. Why can we not simply split LayerNorm activations along the hidden dimension?
2. What fraction of total activation memory do the non-TP activations represent?
3. How does this waste scale with the TP degree $N$?

---

In [ ]:
#@title 🎧 Listen: Solution Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_solution_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
#@title 🎧 Code Walkthrough: Solution Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_solution_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compute_memory_waste(hidden_dim, seq_len, batch_size, num_layers, tp_degree, dtype_bytes=2):
    """Compute the activation memory waste from replicated non-TP activations."""
    b, s, h, N, L = batch_size, seq_len, hidden_dim, tp_degree, num_layers
    
    # Step 1: Count tensors
    num_full_tensors = 5   # 2 LN inputs + 2 dropout inputs + 1 residual
    num_mask_tensors = 2   # 2 dropout masks
    
    # Step 2: Bytes per layer per GPU
    elements_per_tensor = b * s * h
    non_tp_bytes_per_layer = (num_full_tensors * elements_per_tensor * dtype_bytes +
                              num_mask_tensors * elements_per_tensor * 1)  # masks are 1 byte
    
    # Step 3: Total across all layers
    non_tp_bytes_total = non_tp_bytes_per_layer * L
    
    # Step 4: Ideal if distributed
    ideal_bytes_total = non_tp_bytes_total // N
    
    # Step 5: Waste
    wasted_bytes = non_tp_bytes_total - ideal_bytes_total
    
    return {
        'non_tp_per_gpu_per_layer_mb': bytes_to_mb(non_tp_bytes_per_layer),
        'non_tp_per_gpu_total_mb': bytes_to_mb(non_tp_bytes_total),
        'ideal_per_gpu_total_mb': bytes_to_mb(ideal_bytes_total),
        'wasted_per_gpu_mb': bytes_to_mb(wasted_bytes),
    }


# Verify with our config
result = compute_memory_waste(HIDDEN_DIM, SEQ_LEN, BATCH_SIZE, 40, TP_DEGREE)
print(f"Non-TP memory per GPU per layer: {result['non_tp_per_gpu_per_layer_mb']:.2f} MB")
print(f"Non-TP memory per GPU total (40 layers): {result['non_tp_per_gpu_total_mb']:.2f} MB")
print(f"Ideal (distributed):  {result['ideal_per_gpu_total_mb']:.2f} MB")
print(f"Wasted per GPU:       {result['wasted_per_gpu_mb']:.2f} MB")

In [ ]:
#@title 🎧 Transition: Scaling Analysis Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_scaling_analysis_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Scaling Analysis: How Bad Is the Waste at Different Model Sizes?

In [ ]:
#@title 🎧 Code Walkthrough: Scaling Table Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_scaling_table_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Compare waste across model sizes (from the article)
configs = [
    {'name': '7B',   'h': 4096,  'layers': 32, 'heads': 32},
    {'name': '13B',  'h': 5120,  'layers': 40, 'heads': 40},
    {'name': '30B',  'h': 7168,  'layers': 60, 'heads': 56},
    {'name': '70B',  'h': 8192,  'layers': 80, 'heads': 64},
    {'name': '175B', 'h': 12288, 'layers': 96, 'heads': 96},
]

tp_degrees = [2, 4, 8]

print(f"{'Model':<8} {'TP=2 waste (GB)':<18} {'TP=4 waste (GB)':<18} {'TP=8 waste (GB)':<18}")
print("-" * 65)

results_for_plot = {}

for cfg in configs:
    row = f"{cfg['name']:<8}"
    for N in tp_degrees:
        r = compute_memory_waste(cfg['h'], 2048, 1, cfg['layers'], N)
        waste_gb = r['wasted_per_gpu_mb'] / 1024
        row += f"{waste_gb:<18.1f}"
        results_for_plot[(cfg['name'], N)] = waste_gb
    print(row)

In [ ]:
#@title 🎧 What to Look For: Scaling Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_scaling_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualization: waste vs model size for different TP degrees

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

model_names = [c['name'] for c in configs]
x = np.arange(len(model_names))
width = 0.25

colors = ['#2196F3', '#FF9800', '#F44336']

for i, N in enumerate(tp_degrees):
    wastes = [results_for_plot[(name, N)] for name in model_names]
    bars = ax.bar(x + i * width, wastes, width, label=f'TP = {N}',
                  color=colors[i], alpha=0.85)
    # Add value labels
    for bar, val in zip(bars, wastes):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', fontsize=8)

ax.set_xlabel('Model Size', fontsize=12)
ax.set_ylabel('Wasted Memory per GPU (GB)', fontsize=12)
ax.set_title('Activation Memory Waste from Replicated LayerNorm/Dropout\n(s=2048, b=1, BF16)',
             fontsize=13)
ax.set_xticks(x + width)
ax.set_xticklabels(model_names)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add A100 80GB reference line
ax.axhline(y=80, color='red', linestyle='--', alpha=0.5, label='A100 80GB total')
ax.text(len(model_names) - 0.5, 81, 'A100 80GB', color='red', fontsize=9, alpha=0.7)

plt.tight_layout()
plt.show()

print("\nKey takeaway: For a 175B model with TP=8, each GPU wastes ~18 GB")
print("just on replicated LayerNorm and dropout activations!")
print("This is memory that sequence parallelism can recover.")

In [ ]:
#@title 🎧 Transition: Live Measurement Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_live_measurement_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Live Measurement: Actual GPU Memory with PyTorch

Let us go beyond theoretical calculations and actually measure the GPU memory consumed by a simulated forward pass, separating TP and non-TP activations.

In [ ]:
#@title 🎧 Code Walkthrough: Live Measurement Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_live_measurement_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Measure actual GPU memory for TP vs non-TP activations
# We use smaller dims to fit in Colab T4

h_test = 1024
s_test = 512
b_test = 1
N_test = 4

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# Simulate storing non-TP activations (what standard TP does)
non_tp_tensors = []

if torch.cuda.is_available():
    baseline = torch.cuda.memory_allocated()
    
    # 5 full-size activation tensors + 2 masks
    for i in range(5):
        non_tp_tensors.append(torch.randn(b_test, s_test, h_test, 
                                           dtype=torch.bfloat16, device='cuda'))
    for i in range(2):
        non_tp_tensors.append(torch.randint(0, 2, (b_test, s_test, h_test),
                                             dtype=torch.uint8, device='cuda'))
    
    non_tp_actual = torch.cuda.memory_allocated() - baseline
    
    # Now simulate what SP would store (split by N)
    del non_tp_tensors
    torch.cuda.empty_cache()
    
    baseline = torch.cuda.memory_allocated()
    sp_tensors = []
    for i in range(5):
        sp_tensors.append(torch.randn(b_test, s_test // N_test, h_test,
                                       dtype=torch.bfloat16, device='cuda'))
    for i in range(2):
        sp_tensors.append(torch.randint(0, 2, (b_test, s_test // N_test, h_test),
                                         dtype=torch.uint8, device='cuda'))
    
    sp_actual = torch.cuda.memory_allocated() - baseline
    
    del sp_tensors
    torch.cuda.empty_cache()
    
    print(f"Measured GPU memory (h={h_test}, s={s_test}, N={N_test}):")
    print(f"  Standard TP (non-TP activations): {bytes_to_mb(non_tp_actual):.2f} MB per GPU")
    print(f"  With SP (split activations):      {bytes_to_mb(sp_actual):.2f} MB per GPU")
    print(f"  Savings:                          {bytes_to_mb(non_tp_actual - sp_actual):.2f} MB ({non_tp_actual / sp_actual:.1f}x)")
else:
    # CPU fallback: compute theoretically
    non_tp_theoretical = 5 * b_test * s_test * h_test * 2 + 2 * b_test * s_test * h_test * 1
    sp_theoretical = 5 * b_test * (s_test // N_test) * h_test * 2 + 2 * b_test * (s_test // N_test) * h_test * 1
    print(f"Theoretical memory (h={h_test}, s={s_test}, N={N_test}):")
    print(f"  Standard TP (non-TP activations): {bytes_to_mb(non_tp_theoretical):.2f} MB per GPU")
    print(f"  With SP (split activations):      {bytes_to_mb(sp_theoretical):.2f} MB per GPU")
    print(f"  Savings:                          {bytes_to_mb(non_tp_theoretical - sp_theoretical):.2f} MB ({non_tp_theoretical / sp_theoretical:.1f}x)")

In [ ]:
#@title 🎧 Listen: Key Insight Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_key_insight_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. The Key Insight: Where Can We Split?

We have established that:
- LayerNorm needs the **full hidden dimension** (cannot split along $h$)
- Dropout is element-wise (does not care about structure)
- Residual addition is element-wise

But LayerNorm computes statistics **independently for each token**. There is no interaction between token positions. This means we **can** split along the **sequence dimension**!

Each GPU processes $s/N$ tokens with the full hidden dimension $h$. This is exactly what sequence parallelism does, and we will implement it in the next notebook.

In [ ]:
#@title 🎧 Code Walkthrough: Sequence Split Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_sequence_split_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verify: LayerNorm IS correct when split along the sequence dimension

h = 16
s = 8
N = 4
x = torch.randn(1, s, h)

ln = nn.LayerNorm(h)
with torch.no_grad():
    ln.weight.fill_(1.0)
    ln.bias.fill_(0.0)

# Correct: full LayerNorm
correct = ln(x)

# Split along sequence, apply LayerNorm to each chunk
chunks = x.split(s // N, dim=1)  # 4 chunks of (1, 2, 16)
chunk_outputs = [ln(chunk) for chunk in chunks]
reconstructed = torch.cat(chunk_outputs, dim=1)

max_error = (correct - reconstructed).abs().max().item()
print(f"Max error (split along sequence): {max_error:.10f}")
print(f"\nSplitting along the SEQUENCE dimension gives the EXACT same result!")
print(f"This is the foundation of sequence parallelism.")
print(f"\nEach of {N} GPUs processes {s // N} tokens with full h={h} features.")
print(f"Memory per GPU: (1, {s // N}, {h}) instead of (1, {s}, {h})")
print(f"Reduction: {N}x")

In [ ]:
#@title 🎧 Wrap-Up: Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_24_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 10. Summary

In this notebook, we discovered and quantified a critical memory inefficiency in tensor parallelism:

**The Problem:**
- Tensor parallelism splits MLP and attention activations across GPUs: $(b, s, h/N)$ per GPU
- But LayerNorm requires the **full hidden dimension**, so its input must be $(b, s, h)$ on every GPU
- Dropout and residual operations also receive the full tensor
- These non-TP activations are **replicated** on every GPU, wasting $N-1$ copies

**The Waste:**
- For a 175B model with TP=8: roughly 20 GB wasted per GPU
- Non-TP activations can represent ~30% of total activation memory

**The Insight:**
- LayerNorm, dropout, and residual are **independent across tokens**
- We CAN split along the **sequence dimension** without changing the result
- Each GPU processes $s/N$ tokens with full $h$ features

In the next notebook, we will implement sequence parallelism and show that this split can be achieved with **zero additional communication cost**.